<a href="https://colab.research.google.com/github/LaraDondossola/Classificacao-eventos-climaticos/blob/main/Notebooks/Interface-eventos-clim%C3%A1ticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q streamlit joblib pandas scikit-learn
!npm install -g localtunnel
!pip install -q pyngrok streamlit

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 1s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [19]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path

# Configuração da página
st.set_page_config(
    page_title="Predição de Eventos Climáticos",
    page_icon="⛈️",
    layout="wide"
)

# --- CSS CUSTOMIZADO PARA OCULTAR OS BOTÕES DE INCREMENTO E DECREMENTO (- e +) ---
st.markdown("""
    <style>
    /* Oculta os botões do Streamlit */
    button[data-testid="stNumberInputStepDown"],
    button[data-testid="stNumberInputStepUp"] {
        display: none !important;
    }
    /* Oculta as setas nativas do navegador para inputs numéricos */
    input[type=number]::-webkit-inner-spin-button,
    input[type=number]::-webkit-outer-spin-button {
        -webkit-appearance: none;
        margin: 0;
    }
    input[type=number] {
        -moz-appearance: textfield;
    }
    </style>
""", unsafe_allow_html=True)

# Título e Descrição
st.title("⛈️ Painel de Avaliação de Impacto e Risco Climático")
st.markdown("Insira os dados da ocorrência abaixo para calcular o **Nível de Risco** e estimar a **População Afetada**.")

# --- CARREGAMENTO DOS MODELOS ---
@st.cache_resource
def carregar_artefatos():
    base_path = Path("Models") if Path("Models").exists() else Path(".")

    preprocessor = joblib.load(base_path / "preprocessor.joblib")
    modelo_clf = joblib.load(base_path / "modelo_clf_rf.joblib")
    modelo_reg = joblib.load(base_path / "modelo_reg_rf.joblib")

    return preprocessor, modelo_clf, modelo_reg

try:
    preprocessor, modelo_clf, modelo_reg = carregar_artefatos()
    st.sidebar.success("✅ Modelos carregados com sucesso!")
except Exception as e:
    st.error(f"Erro ao carregar os modelos: {e}")
    st.stop()

# --- FORMULÁRIO DE ENTRADA DE DADOS ---
st.header("📋 Dados da Ocorrência")

# Mapeamento de Meses
meses_map = {
    "Janeiro": 1, "Fevereiro": 2, "Março": 3, "Abril": 4,
    "Maio": 5, "Junho": 6, "Julho": 7, "Agosto": 8,
    "Setembro": 9, "Outubro": 10, "Novembro": 11, "Dezembro": 12
}

col1, col2, col3 = st.columns(3)

with col1:
    uf = st.selectbox("Estado (UF)", options=[
        'AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA',
        'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN',
        'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO'
    ], index=23) # Padrão SC

    populacao = st.number_input("População Total do Município", min_value=0, value=25000)

with col2:
    nome_mes = st.selectbox("Mês do Registro", options=list(meses_map.keys()), index=5)
    mes = meses_map[nome_mes]

    trimestre = (mes - 1) // 3 + 1
    st.info(f"🗓️ Trimestre estimado: **{trimestre}º Trimestre**")

with col3:
    hab_danificadas = st.number_input("Habitações Danificadas", min_value=0, value=0)
    hab_destruidas = st.number_input("Habitações Destruídas", min_value=0, value=0)
    infra_danificada = st.number_input("Obras de Infraestrutura Pública Danificadas", min_value=0, value=0)

# --- BOTÃO DE PREDIÇÃO E PROCESSAMENTO ---
st.markdown("---")

if st.button("🚀 Calcular Previsões", type="primary", use_container_width=True):
    dados_entrada = pd.DataFrame([{
        'UF': uf,
        'População': populacao,
        'Mes_Registro': mes,
        'Trimestre': trimestre,
        'DM_Unidades Habitacionais Danificadas': hab_danificadas,
        'DM_Unidades Habitacionais Destruídas': hab_destruidas,
        'DM_Obras de infraestrutura pública Danificadas': infra_danificada
    }])

    try:
        dados_processados = preprocessor.transform(dados_entrada)
        pred_risco = modelo_clf.predict(dados_processados)[0]

        # Predição da População Afetada (Regressão)
        pred_pop_afetada = modelo_reg.predict(dados_processados)[0]
        pred_pop_afetada = max(0, int(round(pred_pop_afetada)))

        res_col1, res_col2 = st.columns(2)

        with res_col1:
            st.subheader("🔴 Nível de Risco Classificado")
            st.metric(label="Risco Estimado", value=str(pred_risco))

        with res_col2:
            st.subheader("👥 População Afetada Estimada")
            st.metric(label="Total de Pessoas Afetadas", value=f"{pred_pop_afetada:,} pessoas".replace(",", "."))

    except Exception as err:
        st.error(f"Erro ao processar as informações: {err}")

Overwriting app.py


In [20]:
from pyngrok import ngrok
import os

# 1. Configurar o seu Authtoken (substitua com o token do site)
ngrok.set_auth_token("3Eu0uGhV8Sw9rvvCGXlcwhNKIfj_81kRpesKtmutVx3YxxhhF")

# 2. Encerrar conexões anteriores
ngrok.kill()

# 3. Criar o túnel na porta 8501
public_url = ngrok.connect(8501)
print(f"🔗 Acesse sua aplicação sem erros aqui: {public_url}")

# 4. Rodar o Streamlit
!streamlit run app.py --server.port 8501

🔗 Acesse sua aplicação sem erros aqui: NgrokTunnel: "https://slideshow-litter-outspoken.ngrok-free.dev" -> "http://localhost:8501"




2026-09-10 11:55:17.766 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.66.114.22:8501

  Stopping...
  Stopping...
